# MewtwoLLM — Full Training Pipeline
## Train a 40M param LLM from scratch on free T4 GPU

**Architecture:** RoPE + RMSNorm + SwiGLU + GQA

**Pipeline:** Scrape → Tokenize → Pretrain → SFT → DPO → RLHF → Eval

**Runtime:** Change to GPU: Runtime → Change runtime type → T4 GPU

## Step 0: Setup — Clone repo + install deps

In [ ]:
#@title Clone MewtwoLLM repo { display-mode: "form" }
!git clone https://github.com/superduperpiyuxh/MewtwoLLM.git
%cd MewtwoLLM
!pip install torch --index-url https://download.pytorch.org/whl/cpu sentencepiece scrapling huggingface_hub datasets --quiet

In [ ]:
#@title Verify GPU { display-mode: "form" }
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be slow!')

## Step 1: Scrape training data

In [ ]:
#@title Scrape Wikipedia + arXiv + GitHub { display-mode: "form" }
import os, sys
sys.path.insert(0, '.')

from src.data.scraper import WebScraper

os.makedirs('data/raw', exist_ok=True)
scraper = WebScraper(output_dir='data/raw')

# Wikipedia
wiki_urls = [
    'https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)',
    'https://en.wikipedia.org/wiki/Large_language_model',
    'https://en.wikipedia.org/wiki/Attention_(machine_learning)',
    'https://en.wikipedia.org/wiki/GPT-3',
    'https://en.wikipedia.org/wiki/BERT_(language_model)',
    'https://en.wikipedia.org/wiki/Fine-tuning_(deep_learning)',
    'https://en.wikipedia.org/wiki/Reinforcement_learning_from_human_feedback',
]

# arXiv
arxiv_urls = [
    'https://arxiv.org/abs/1706.03762',  # Attention Is All You Need
    'https://arxiv.org/abs/2005.14165',  # GPT-3
    'https://arxiv.org/abs/2107.03374',  # SwiGLU
    'https://arxiv.org/abs/2305.18290',  # LLaMA
]

all_urls = wiki_urls + arxiv_urls
scraper.scrape_urls(all_urls)
print(f'Scraped {len(all_urls)} pages → data/raw/')

## Step 2: Download FineWeb-Edu from HuggingFace

In [ ]:
#@title Download FineWeb-Edu sample { display-mode: "form" }
from huggingface_hub import hf_hub_download
import os

os.makedirs('data/raw', exist_ok=True)

# Download 100K row sample
print('Downloading FineWeb-Edu-100k...')
path = hf_hub_download(
    repo_id='HuggingFaceFW/fineweb-edu',
    filename='sample/100k.txt',
    repo_type='dataset',
    local_dir='data/raw',
)
print(f'Downloaded to: {path}')

# Show file size
size_mb = os.path.getsize(path) / 1e6
print(f'Size: {size_mb:.1f} MB')

## Step 3: Train SentencePiece tokenizer

In [ ]:
#@title Train tokenizer on collected data { display-mode: "form" }
import os, glob

# Combine all text for tokenizer training
all_files = glob.glob('data/raw/**/*', recursive=True)
all_files = [f for f in all_files if os.path.isfile(f) and not f.endswith(('.bin', '.npy'))]

combined = []
for fpath in sorted(all_files):
    try:
        with open(fpath, 'r', errors='ignore') as f:
            text = f.read()
            if len(text.strip()) > 100:
                combined.append(text)
    except:
        pass

with open('data/tokenizer_input.txt', 'w') as f:
    f.write('\n'.join(combined))

print(f'Tokenizer corpus: {len(combined)} chunks, {sum(len(c) for c in combined):,} chars')

# Train SentencePiece
from src.tokenizer.tokenizer import MewtwoTokenizer

os.makedirs('data/tokenizer', exist_ok=True)
tokenizer = MewtwoTokenizer(vocab_size=32000)
tokenizer.train(input_file='data/tokenizer_input.txt', model_prefix='data/tokenizer/mewtwo')

# Verify
test = 'Hello, this is a test of the MewtwoLLM tokenizer!'
tokens = tokenizer.encode(test)
print(f'\nTest: "{test}"')
print(f'Tokens: {tokens}')
print(f'Decoded: "{tokenizer.decode(tokens)}"')
print(f'Vocab size: {tokenizer.vocab_size}')

## Step 4: Pretrain MewtwoLLM on GPU

In [ ]:
#@title Pretrain (1000 steps on T4 GPU) { display-mode: "form" }
import torch
import sys, os

sys.path.insert(0, '.')
from config.model_config import MewtwoConfig
from src.model.pocketllm import MewtwoLLM
from src.tokenizer.tokenizer import MewtwoTokenizer
from src.training.pretrain import Pretrainer

config = MewtwoConfig()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = MewtwoLLM(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params ({n_params/1e6:.1f}M)')

tokenizer = MewtwoTokenizer.from_pretrained('data/tokenizer')

# Collect training texts
train_data = []
for fname in os.listdir('data/raw'):
    fpath = os.path.join('data/raw', fname)
    if os.path.isfile(fpath) and not fname.startswith('.'):
        try:
            with open(fpath, 'r', errors='ignore') as f:
                text = f.read()
                if len(text.strip()) > 100:
                    train_data.append(text)
        except:
            pass

print(f'Training data: {len(train_data)} chunks')

# Train
trainer = Pretrainer(
    config=config,
    tokenizer=tokenizer,
    output_dir='checkpoints/pretrain',
    lr=3e-4,
    warmup_steps=100,
    max_steps=1000,
    batch_size=8,
    gradient_accumulation_steps=2,
    log_every=10,
    eval_every=100,
    save_every=500,
)
trainer.train(train_data)
print('Pretraining complete!')

## Step 5: Supervised Fine-Tuning

In [ ]:
#@title SFT on instruction dataset { display-mode: "form" }
import json, os, sys, torch

sys.path.insert(0, '.')
from config.model_config import MewtwoConfig
from src.model.pocketllm import MewtwoLLM
from src.tokenizer.tokenizer import MewtwoTokenizer
from src.alignment.sft import SFTTrainer

# Create instruction dataset
os.makedirs('data/instruction', exist_ok=True)
instructions = [
    {'instruction': 'Explain what a transformer is in ML.', 'input': '', 'output': 'A transformer is a neural network architecture using self-attention, introduced in Attention Is All You Need (2017). It processes all input tokens in parallel, computing attention scores to weigh the importance of each token relative to others.'},
    {'instruction': 'What is DPO?', 'input': '', 'output': 'Direct Preference Optimization directly optimizes a language model using human preference data, skipping the separate reward model used in RLHF. It uses a loss function derived from the Bradley-Terry preference model.'},
    {'instruction': 'What does RoPE do?', 'input': '', 'output': 'Rotary Position Embedding encodes position by rotating query and key vectors. Attention scores depend on relative distance between tokens, enabling better length generalization.'},
    {'instruction': 'Why RMSNorm over LayerNorm?', 'input': '', 'output': 'RMSNorm removes mean centering, normalizing only by root mean square. This is 10-15% faster with equivalent performance, and eliminates the bias term.'},
    {'instruction': 'What is SwiGLU?', 'input': '', 'output': 'SwiGLU combines Swish activation with a Gated Linear Unit in the FFN. It uses three weight matrices instead of two, where the gate controls information flow. Consistently outperforms ReLU and GELU.'},
]
with open('data/instruction/instructions.jsonl', 'w') as f:
    for item in instructions:
        f.write(json.dumps(item) + '\n')

config = MewtwoConfig()
tokenizer = MewtwoTokenizer.from_pretrained('data/tokenizer')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

sft = SFTTrainer(
    config=config,
    tokenizer=tokenizer,
    output_dir='checkpoints/sft',
    lr=5e-5,
    max_steps=500,
    batch_size=4,
    gradient_accumulation_steps=4,
)
sft.train('data/instruction/instructions.jsonl')
print('SFT complete!')

## Step 6: DPO Alignment

In [ ]:
#@title DPO on preference data { display-mode: "form" }
import json, os, sys, torch

sys.path.insert(0, '.')
from config.model_config import MewtwoConfig
from src.model.pocketllm import MewtwoLLM
from src.tokenizer.tokenizer import MewtwoTokenizer
from src.alignment.dpo import DPOTrainer

# Create preference dataset
os.makedirs('data/preference', exist_ok=True)
preferences = [
    {'prompt': 'What is ML?', 'chosen': 'Machine learning is a subset of AI where systems learn patterns from data to make predictions without being explicitly programmed. Types include supervised, unsupervised, and reinforcement learning.', 'rejected': 'idk lol its when computers learn stuff'},
    {'prompt': 'Explain gradient descent.', 'chosen': 'Gradient descent iteratively adjusts parameters to minimize loss by computing the gradient and moving opposite to it. The learning rate controls step size.', 'rejected': 'gradient descent is when you go downhill on a graph'},
    {'prompt': 'What is attention?', 'chosen': 'Attention computes weighted sums of input tokens using query-key-value pairs. Attention scores are dot products of query and key, scaled and softmaxed.', 'rejected': 'attention is when the model pays attention to words'},
]
with open('data/preference/preferences.jsonl', 'w') as f:
    for item in preferences:
        f.write(json.dumps(item) + '\n')

config = MewtwoConfig()
tokenizer = MewtwoTokenizer.from_pretrained('data/tokenizer')

dpo = DPOTrainer(
    config=config,
    tokenizer=tokenizer,
    output_dir='checkpoints/dpo',
    lr=5e-6,
    max_steps=200,
    batch_size=2,
    gradient_accumulation_steps=4,
    beta=0.1,
)
dpo.train('data/preference/preferences.jsonl')
print('DPO complete!')

## Step 7: Push checkpoints to HuggingFace Hub

In [ ]:
#@title Upload model to HF Hub { display-mode: "form" }
from huggingface_hub import HfApi, login
import os

# Login (you'll need a HF token)
HF_TOKEN = ''  #@param {type:"string"}
REPO_NAME = 'superduperpiyuxh/MewtwoLLM-40M'  #@param {type:"string"}

if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged in to HuggingFace')
else:
    print('Set HF_TOKEN above to upload. Get token at https://huggingface.co/settings/tokens')

#@title Or run this to login interactively
#!huggingface-cli login

api = HfApi()

# Create repo
try:
    api.create_repo(repo_id=REPO_NAME, exist_ok=True)
    print(f'Repo: https://huggingface.co/{REPO_NAME}')
except Exception as e:
    print(f'Repo creation: {e}')

# Upload best checkpoint
for stage in ['dpo', 'sft', 'pretrain']:
    ckpt_path = f'checkpoints/{stage}/best_model.pt'
    if os.path.exists(ckpt_path):
        print(f'Uploading {stage} checkpoint...')
        api.upload_file(
            path_or_fileobj=ckpt_path,
            path_in_repo=f'{stage}/best_model.pt',
            repo_id=REPO_NAME,
        )
        break

# Upload tokenizer
for fname in os.listdir('data/tokenizer'):
    fpath = os.path.join('data/tokenizer', fname)
    if os.path.isfile(fpath):
        api.upload_file(
            path_or_fileobj=fpath,
            path_in_repo=f'tokenizer/{fname}',
            repo_id=REPO_NAME,
        )

# Upload config
api.upload_file(
    path_or_fileobj='config/model_config.py',
    path_in_repo='config/model_config.py',
    repo_id=REPO_NAME,
)

print(f'\nDone! Model at: https://huggingface.co/{REPO_NAME}')

## Step 8: Generate text

In [ ]:
#@title Generate with trained model { display-mode: "form" }
import torch, sys

sys.path.insert(0, '.')
from config.model_config import MewtwoConfig
from src.model.pocketllm import MewtwoLLM
from src.tokenizer.tokenizer import MewtwoTokenizer
from src.inference.generate import generate

prompt = "What is the meaning of life?"  #@param {type:"string"}
max_tokens = 200  #@param {type:"integer"}
temperature = 0.8  #@param {type:"number"}

config = MewtwoConfig()
tokenizer = MewtwoTokenizer.from_pretrained('data/tokenizer')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = MewtwoLLM(config).to(device)

# Load best checkpoint
import os
for stage in ['dpo', 'sft', 'pretrain']:
    ckpt = f'checkpoints/{stage}/best_model.pt'
    if os.path.exists(ckpt):
        checkpoint = torch.load(ckpt, map_location=device, weights_only=False)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        print(f'Loaded: {ckpt}')
        break

model.eval()
print(f'\nPrompt: {prompt}')
print('=' * 60)

output = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=max_tokens,
    temperature=temperature,
    top_k=50,
    top_p=0.95,
)
print(output)
print('=' * 60)